# Indic Parler-TTS listening test (Storyboard narration)
Free, open-source (Apache-2.0) model **ai4bharat/indic-parler-tts**. This notebook only makes test audio for listening; nothing is published.

**Before you start**
1. Runtime → Change runtime type → **T4 GPU** (free).
2. Open https://huggingface.co/ai4bharat/indic-parler-tts while signed in to Hugging Face and **accept the access conditions** (shares your Hugging Face contact info with the model owners).
3. Create a **Read** token at https://huggingface.co/settings/tokens — you will paste it yourself in step 2 below (never share it).

Then: Runtime → **Run all**. Listen to the players at the bottom.

Recipe being tested (same as the approved narration recipe): story text, **800 ms pause**, then the question. Choices are not read. Asterisks removed in the narration input only.


In [ ]:
# 1. Use PyTorch only: stop transformers from loading Colab's preinstalled TensorFlow
#    (Parler-TTS installs an older protobuf that TensorFlow cannot import). Must run before step 4.
import os
os.environ['USE_TF'] = '0'
os.environ['USE_TORCH'] = '1'
os.environ['TRANSFORMERS_NO_TF'] = '1'

# Check the GPU
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch Runtime to T4 GPU')


In [ ]:
# 2. Install Parler-TTS (inside this Colab runtime only)
!pip -q install git+https://github.com/huggingface/parler-tts.git soundfile

In [ ]:
# 3. Sign in to Hugging Face (the model is gated). This cell WAITS until a token is available.
#    Option A: Colab key icon (left sidebar) -> add secret HF_TOKEN = your Read token, turn on "Notebook access".
#    Option B: no secret -> a hidden box appears below; paste your Read token and press Enter.
from huggingface_hub import login, whoami
token = None
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    token = None
if not token:
    from getpass import getpass
    token = getpass('Paste your Hugging Face Read token (hidden), then press Enter: ').strip()
login(token=token, add_to_git_credential=False)
print('Signed in to Hugging Face as:', whoami()['name'])


In [ ]:
# 4. Load the model
import time, re, numpy as np, soundfile as sf
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer, set_seed
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
model = ParlerTTSForConditionalGeneration.from_pretrained('ai4bharat/indic-parler-tts').to(device)
tokenizer = AutoTokenizer.from_pretrained('ai4bharat/indic-parler-tts')
description_tokenizer = AutoTokenizer.from_pretrained(model.config.text_encoder._name_or_path)
SR = model.config.sampling_rate
print('loaded; sampling rate', SR)

In [ ]:
# 5. Helpers: speak sentence by sentence, join with short pauses (the model card gives no max length)
def sentences(text):
    flat = text.replace('\n', ' ').strip()
    parts = re.split(r'(?<=[.!?।])\s+|(?<=[.!?।]["”])\s+', flat)
    return [p.strip() for p in parts if p.strip()]

def describe(speaker):
    return (f"{speaker}'s voice is warm and expressive, telling a story to children at a moderate pace, "
            "with very clear audio and no background noise.")

def say(text, speaker, seed=0):
    set_seed(seed)
    d = description_tokenizer(describe(speaker), return_tensors='pt').to(device)
    p = tokenizer(text, return_tensors='pt').to(device)
    g = model.generate(input_ids=d.input_ids, attention_mask=d.attention_mask,
                       prompt_input_ids=p.input_ids, prompt_attention_mask=p.attention_mask)
    return g.cpu().numpy().squeeze()

def narrate(text, question, speaker, gap_s=0.25, question_gap_s=0.8):
    t0 = time.time(); out = []
    for s in sentences(text):
        out += [say(s, speaker), np.zeros(int(SR * gap_s))]
    if question:
        out[-1] = np.zeros(int(SR * question_gap_s))
        out.append(say(question, speaker))
    audio = np.concatenate(out)
    secs = len(audio) / SR
    print(f'{speaker}: {secs:.1f} s of audio made in {time.time() - t0:.1f} s')
    return audio


In [ ]:
# 6. Real story text: blue-jackal / start (English and Hindi)
EN_TEXT = "In the forest outside a small town, there lived a jackal named Chandarava. He was clever and quick — but also always hungry, always scheming, always looking for an easier life.\n\nOne evening, chased by a pack of dogs into the town, he stumbled and fell — SPLASH — into a large vat of blue dye outside a cloth merchant's house.\n\nThe dogs sniffed at the vat, then backed away in confusion. They had never seen anything like this before. They ran off, yelping.\n\nChandarava climbed out, dripping. He looked down at himself.\n\nHe was blue. Completely, brilliantly, impossibly blue. No such animal existed in the world.\n\nHe walked to a still puddle and stared at his reflection for a long, long time.\n\n\"No one,\" he thought slowly, \"has ever seen anything like me.\""
EN_QUESTION = "You are soaking wet, bright blue, and every animal who sees you stumbles back in shock. A wild thought crosses your mind. What do you do?"
HI_TEXT = "एक छोटे से शहर के बाहर जंगल में चंदरव नाम का एक सियार रहता था। वह चालाक और फुर्तीला था — लेकिन हमेशा भूखा, हमेशा चालें सोचता, हमेशा आसान ज़िंदगी की तलाश में।\n\nएक शाम, कुत्तों के एक झुंड से भागते हुए शहर में घुसा, लड़खड़ाया और — छपाक — एक कपड़े के व्यापारी के घर के बाहर रखे नीले रंग के बड़े कुंड में गिर गया।\n\nकुत्तों ने कुंड को सूँघा, फिर उलझन में पीछे हट गए। उन्होंने ऐसा कुछ कभी नहीं देखा था। वे भौंकते हुए भाग गए।\n\nचंदरव टपकता हुआ बाहर निकला। उसने खुद को नीचे देखा।\n\nवह नीला था। पूरी तरह, चमकीला, असंभव रूप से नीला। दुनिया में ऐसा कोई जानवर नहीं था।\n\nवह एक शांत पोखरे के पास गया और बहुत देर तक अपना प्रतिबिंब देखता रहा।\n\n\"किसी ने,\" उसने धीरे-धीरे सोचा, \"ऐसा कुछ कभी नहीं देखा।\""
HI_QUESTION = "तुम पूरी तरह भीगे हो, चमकीले नीले हो, और जो भी जानवर तुम्हें देखता है घबराकर पीछे हट जाता है। एक पागल विचार मन में आता है। तुम क्या करते हो?"


In [ ]:
# 7. Make the four test files (two English voices, two Hindi voices)
from IPython.display import Audio, display
tests = [('en', 'Thoma', EN_TEXT, EN_QUESTION), ('en', 'Mary', EN_TEXT, EN_QUESTION),
         ('hi', 'Rohit', HI_TEXT, HI_QUESTION), ('hi', 'Divya', HI_TEXT, HI_QUESTION)]
for lang, speaker, text, q in tests:
    audio = narrate(text, q, speaker)
    path = f'blue-jackal_start_{lang}_{speaker}.wav'
    sf.write(path, audio, SR)
    print(path); display(Audio(audio, rate=SR))

**What to listen for:** natural Hindi and Indian-English pronunciation of names (Chandarava), smooth joins between sentences, the 800 ms pause before the question, no glitches or skipped words. Note the 'made in X s' timings — they tell us how long all 190 files would take.

In [ ]:
# 8. (Optional) Download the four WAV files as one zip
import zipfile, glob
from google.colab import files
with zipfile.ZipFile('indic_parler_test.zip', 'w') as z:
    for f in glob.glob('blue-jackal_start_*.wav'):
        z.write(f)
files.download('indic_parler_test.zip')
